# Gradient Boosting (Pipeline B)

**Model**: `GradientBoostingRegressor` / **Target**: gdpc1
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 6
**Scaling**: NO / **HP tuning**: YES / **10-model averaging**: YES
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025


In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
                     get_features, load_data, PREDICTIONS_DIR, TARGET)

SEED=42; np.random.seed(SEED)
TRAIN_START='1959-01-01'; TRAIN_END='2007-12-31'; VAL_END='2016-12-31'
TEST_START='2017-01-01'; TEST_END='2025-12-31'
N_LAGS=4; N_MODELS=10
VINTAGES={'m1':-2,'m2':-1,'m3':0,'post1':1}
print('GB â€” Cat3, n_lags=4, 10-model avg')

GB â€” Cat3, n_lags=4, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print(f'Features: {len(features)}')


Features: 56


In [3]:
# Phase A: HP tuning on train+val (1959-2016) via TimeSeriesSplit
print('Phase A: HP tuning on train+val (1959-2016)...')

tune_data = monthly[(monthly['date']>=TRAIN_START)&(monthly['date']<=VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]),:].dropna(axis=0,how='any').reset_index(drop=True)
feature_cols = [c for c in tune_flat.columns if c!='date' and c!=TARGET and any(c==f or c.startswith(f+'_') for f in features)]

tscv = TimeSeriesSplit(n_splits=5)
best_score = np.inf
best_params = {'learning_rate': 0.1, 'n_estimators': 200, 'max_depth': 3}

for lr in [0.01, 0.05, 0.1]:
    for ne in [200, 500]:
        for md in [3, 5]:
            scores = []
            for tr, va in tscv.split(tune_flat):
                m = GradientBoostingRegressor(loss='squared_error', learning_rate=lr,
                    n_estimators=ne, max_depth=md, max_features='sqrt', random_state=SEED)
                m.fit(tune_flat[feature_cols].values[tr], tune_flat[TARGET].values[tr])
                p = m.predict(tune_flat[feature_cols].values[va])
                scores.append(np.mean((p - tune_flat[TARGET].values[va])**2))
            if np.mean(scores) < best_score:
                best_score = np.mean(scores)
                best_params = {'learning_rate': lr, 'n_estimators': ne, 'max_depth': md}

print(f'Best GB params: {best_params}')


Phase A: HP tuning on train+val (1959-2016)...


Best GB params: {'learning_rate': 0.01, 'n_estimators': 500, 'max_depth': 3}


In [4]:
test_quarters = monthly[(monthly['date']>=TEST_START)&(monthly['date']<=TEST_END)&
                         monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v:[] for v in VINTAGES}; actuals_list=[]; failed=0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date']==pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, fc, N_LAGS
        )

        if len(train_fl)<10: predictions[vn].append(np.nan); continue

        feature_cols = [c for c in train_fl.columns if c != 'date' and c != TARGET and any(c == f or c.startswith(f + '_') for f in features)]
        try:
            models=[]
            for k in range(N_MODELS):
                m=GradientBoostingRegressor(loss='squared_error',
                    learning_rate=best_params['learning_rate'],
                    n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'],
                    max_features='sqrt',random_state=SEED+k)
                m.fit(train_fl[feature_cols].values,train_fl[TARGET].values)
                models.append(m)
            preds=[m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed+=1

    if (i+1)%6==0: print(f'  {i+1} / {len(test_quarters)}')
print(f'Done. {failed} failures.')

  6 / 36


  12 / 36


  18 / 36


  24 / 36


  30 / 36


  36 / 36
Done. 0 failures.


In [5]:
os.makedirs(PREDICTIONS_DIR,exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date':test_quarters,'actual':actuals_list,'prediction':predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR,f'gb_{vn}.csv'),index=False)
    print(f'Saved gb_{vn}.csv')

Saved gb_m1.csv
Saved gb_m2.csv
Saved gb_m3.csv
Saved gb_post1.csv


In [6]:
def rmse(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan
def mae(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan
panels={'Pre-COVID':'2017-01-01,2019-12-31','COVID':'2020-04-01,2021-12-31',
        'Post-COVID':'2022-01-01,2025-12-31','Full':'2017-01-01,2025-12-31'}
a=np.array(actuals_list); d=pd.to_datetime(test_quarters)
for pn,rng in panels.items():
    ps,pe=rng.split(','); m=(d>=ps)&(d<=pe)
    print(f'--- {pn} ---')
    for vn in VINTAGES:
        pp=np.array(predictions[vn])
        print(f'  {vn}  RMSFE={rmse(a[m],pp[m]):.6f}  MAE={mae(a[m],pp[m]):.6f}')

--- Pre-COVID ---
  m1  RMSFE=0.002617  MAE=0.002073
  m2  RMSFE=0.002618  MAE=0.002093
  m3  RMSFE=0.002637  MAE=0.002054
  post1  RMSFE=0.002620  MAE=0.002046
--- COVID ---
  m1  RMSFE=0.042275  MAE=0.026549
  m2  RMSFE=0.041508  MAE=0.025742
  m3  RMSFE=0.042036  MAE=0.025564
  post1  RMSFE=0.041581  MAE=0.025214
--- Post-COVID ---
  m1  RMSFE=0.004471  MAE=0.003353
  m2  RMSFE=0.004691  MAE=0.003579
  m3  RMSFE=0.004618  MAE=0.003564
  post1  RMSFE=0.004904  MAE=0.004065
--- Full ---
  m1  RMSFE=0.019244  MAE=0.007913
  m2  RMSFE=0.018944  MAE=0.007866
  m3  RMSFE=0.019161  MAE=0.007811
  post1  RMSFE=0.018963  MAE=0.007930
